In [1]:
import numpy as np
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
import polars as pl

from ipywidgets import Output, VBox
from IPython.display import display, clear_output

from benchmark_notebook import theme  # noqa: F401  registers thesis template
from benchmark_notebook.io import CloudStorage
from benchmark.scope.filtering import TrunkBatch
from benchmark.scope.profiling import Review, ReviewBatch, Stationarity

In [2]:
issues = pl.from_arrow(
    CloudStorage(bucket="bug-request-bucket").asset(
        "scope-issue-catalog",
        location="main",
        partition="default",
        version="4.0.0",
    )
)

pool = pl.from_arrow(
    CloudStorage(bucket="bug-request-bucket").asset(
        "scope-repository-pool",
        location="main",
        partition="default",
        version="6.0.0",
    )
)

In [3]:
from pathlib import Path

cache = Path("/usr/local/src/.cache")
cache.mkdir(parents=True, exist_ok=True)

# Integration Branch Reconstruction

Given the ordered sequence of base branches of merged PRs $Y_1, \dots, Y_n$, we seek a piecewise constant sequence $Z_1, \dots, Z_n$ (the trunk) that fits the observations while changing only rarely. The reconstruction is defined as a switch-cost minimization problem:

$$
\hat Z_{1:n}
=
\arg\min_{Z_{1:n}}
\left(
\sum_{t=1}^{n} \mathbf{1}[Y_t \neq Z_t]
+
\lambda \sum_{t=2}^{n} \mathbf{1}[Z_t \neq Z_{t-1}]
\right)
$$

The first term penalizes mismatches between observation and trunk. The second term introduces a switch cost: each change of $Z_t$ relative to $Z_{t-1}$ incurs a penalty $\lambda > 0$, enforcing structural stability.

## Switch Penalty Selection

To select the switch penalty $\lambda$ in a data-driven manner, we treat $\lambda$ as a hyperparameter and choose it by predictive performance under a rolling-origin holdout protocol. For a candidate $\lambda$, we fit the trunk on a prefix $Y_{1:t}$ and evaluate how well the inferred trunk generalizes to the subsequent holdout segment $Y_{t+1:t+h}$.

Concretely, let $\hat Z^{(t)}(\lambda)$ denote the inferred trunk sequence on the prefix $Y_{1:t}$, and let $\hat z^{(t)}(\lambda)=\hat Z^{(t)}_t(\lambda)$ be its last trunk state. The holdout risk for split point $t$ is

$$
R_t(\lambda)
=
\frac{1}{h}
\sum_{i=t+1}^{t+h}
\mathbf{1}\!\left[Y_i \neq \hat z^{(t)}(\lambda)\right].
$$

We average this risk over multiple split points (folds) along the sequence,

$$
R(\lambda)
=
\frac{1}{|\mathcal{T}|}
\sum_{t \in \mathcal{T}}
R_t(\lambda),
$$

and select the penalty by empirical risk minimization,

$$
\lambda^*
=
\arg\min_{\lambda}
R(\lambda).
$$

In practice, the minimization is performed over $\lambda$ on a log scale using a black-box optimizer (TPE/Optuna), which is robust to the piecewise-constant and non-smooth dependence of $\hat Z(\lambda)$ on $\lambda$.

## Repository Inclusion Criteria

A repository is considered compatible with the integration-branch model only if the inferred trunk structure is supported by sufficient evidence and exhibits limited structural complexity.

The following conditions are evaluated after estimating the optimal switch penalty $\lambda^*$ via predictive risk minimization.

### Minimal Evidence

The sequence length must exceed a minimum threshold,

$$
n \ge n_{\min}.
$$

Repositories with too few merge events do not provide enough information to identify integration regimes.

### Limited Structural Complexity

Let $m(\lambda^*)$ denote the number of segments in the inferred trunk sequence $\hat Z(\lambda^*)$. The repository is included only if

$$
m(\lambda^*) \le 2.
$$

This constraint ensures that the integration structure is either constant or exhibits at most a single structural regime change. Repositories are included only if both conditions are satisfied. These criteria ensure that the inferred trunk represents a stable integration pattern rather than fragmented or highly oscillatory behavior.

In [4]:
path = cache / "trunk.parquet"
if path.exists():
    trunk = pl.read_parquet(path)
else:
    trunk = TrunkBatch().transform(issues)
    trunk.write_parquet(path)

trunk.filter(pl.col("segments") <= 2)

name,type,number,author,created,closed,labels,payload,trunk,segments
str,str,i64,str,str,str,list[str],struct[5],str,i64
"""spacetelescope/romanisim""","""change""",356,"""yuedong0607""","""2026-04-21T20:31:27Z""","""2026-04-22T19:18:24Z""",[],"{""2026-04-22T19:18:24Z"",""main"",[{""schlafly"",""user"",""APPROVED"",""COLLABORATOR"",""2026-04-22T19:18:14Z""}],5,322}","""main""",1
"""spacetelescope/romanisim""","""change""",355,"""zacharyburnett""","""2026-04-20T19:26:14Z""","""2026-04-20T19:36:16Z""",[],"{""2026-04-20T19:36:16Z"",""main"",[{""braingram"",""user"",""APPROVED"",""CONTRIBUTOR"",""2026-04-20T19:35:13Z""}],3,3}","""main""",1
"""spacetelescope/romanisim""","""change""",353,"""schlafly""","""2026-04-17T19:49:06Z""","""2026-04-20T13:11:10Z""",[],"{""2026-04-20T13:11:10Z"",""main"",[],7,5}","""main""",1
"""spacetelescope/romanisim""","""change""",350,"""schlafly""","""2026-04-16T15:13:56Z""","""2026-04-17T15:19:40Z""",[],"{""2026-04-17T15:19:40Z"",""main"",[],30,8}","""main""",1
"""spacetelescope/romanisim""","""change""",348,"""schlafly""","""2026-04-15T19:17:31Z""","""2026-04-16T17:26:14Z""",[],"{""2026-04-16T17:26:14Z"",""main"",[],3,0}","""main""",1
…,…,…,…,…,…,…,…,…,…
"""red-hat-storage/cephci""","""issue""",2857,"""SrinivasaBharath""","""2023-08-24T08:45:29Z""","""2023-11-23T05:04:22Z""","[""stale-warning-on-inactivity"", ""closed-due-to-inactivity""]",null,null,2
"""red-hat-storage/cephci""","""issue""",2835,"""SrinivasaBharath""","""2023-08-16T00:15:32Z""","""2023-11-14T01:26:33Z""","[""stale-warning-on-inactivity"", ""closed-due-to-inactivity""]",null,null,2
"""red-hat-storage/cephci""","""issue""",2833,"""SrinivasaBharath""","""2023-08-14T08:42:20Z""","""2023-11-12T10:04:13Z""","[""stale-warning-on-inactivity"", ""closed-due-to-inactivity""]",null,null,2


## Review Probability Model

Given an ordered binary sequence of review outcomes $Y_1, \dots, Y_n$ with $Y_t \in \{0,1\}$, we model the review probability over normalized development time.

Let

$$
x_t = \frac{t}{n}
$$

denote the normalized PR index on $[0,1]$. A logistic GAM is fitted:

$$
\text{logit}(p(x)) = f(x),
$$

where $f(x)$ is represented by spline basis functions. The fitted model yields:

- Estimated probability
  $$
  \hat p(x)
  $$

- Confidence interval
  $$
  [\hat p_{\text{lower}}(x), \hat p_{\text{upper}}(x)]
  $$

- Numerical derivative
  $$
  \hat p'(x)
  $$

## Feature Extraction

For a fixed number of segments $m$, we evaluate the functions at quantiles

$$
q_j = \frac{j}{m}, \quad j = 1, \dots, m.
$$

Features consist of:

- $\hat p(q_j)$
- $\hat p'(q_j)$

resulting in a $2m$-dimensional representation of review level and change dynamics.

In [5]:
def profile(
    probability: dict,
    name: str = "",
) -> go.Figure:

    domain = probability["domain"]
    lower = probability["lower"]
    upper = probability["upper"]
    target = probability["target"]
    length = probability["length"]

    raw = np.linspace(0, 1, length)

    title = f"{name} (n={length})" if name else f"Review profile (n={length})"

    return (
        go.Figure()
        .add_trace(
            go.Scatter(
                x=np.concatenate([domain, domain[::-1]]),
                y=np.concatenate([upper, lower[::-1]]),
                fill="toself",
                fillcolor="rgba(0,0,0,0.06)",
                line=dict(width=0),
                name="Interval",
                hoverinfo="skip",
            )
        )
        .add_trace(
            go.Scatter(
                x=raw,
                y=target,
                mode="markers",
                marker=dict(size=3, color="rgba(0,0,0,0.25)"),
                name="Observations",
                # hoverinfo="skip",
            )
        )
        .add_trace(
            go.Scatter(
                x=domain,
                y=probability["probability"],
                mode="lines",
                line=dict(color="#1f77b4", width=1.6),
                name="Estimate",
            )
        )
        .update_layout(
            title=title,
            height=520,
            showlegend=True,
            plot_bgcolor="white",
            paper_bgcolor="white",
            margin=dict(l=60, r=40, t=80, b=60),
        )
        .update_xaxes(
            title_text="Normalized Index",
            range=[0, 1],
            showgrid=False,
            zeroline=False,
            ticks="outside",
        )
        .update_yaxes(
            title_text="Review Probability",
            range=[-0.02, 1.02],
            showgrid=False,
            zeroline=False,
            ticks="outside",
        )
    )

In [6]:
path = cache / "reviews.parquet"
if path.exists():
    reviews = pl.read_parquet(path)
else:
    reviews = ReviewBatch().transform(trunk)
    reviews.write_parquet(path)

reviews

name,n,profile,diagnostics
str,i64,list[list[f64]],struct[6]
"""AMReX-Codes/pyamrex""",373,"[[0.0, 0.002004, … 1.0], [0.401863, 0.403535, … 0.652247], … [9.196383, 9.16332, … 3.15863]]","{true,true,true,3,0.000001,200}"
"""AllenNeuralDynamics/aind-data-…",126,"[[0.0, 0.002004, … 1.0], [0.999999, 0.999999, … 0.931155], … [5.78656, 5.78029, … 28.043874]]","{true,true,true,7,6.2592e-7,200}"
"""ARM-DOE/ACT""",685,"[[0.0, 0.002004, … 1.0], [0.087011, 0.087725, … 0.63749], … [10.369691, 10.356727, … -3.875203]]","{true,true,true,3,0.000003,200}"
"""AMWA-TV/nmos-testing""",462,"[[0.0, 0.002004, … 1.0], [0.041473, 0.048772, … 0.966585], … [124.821954, 124.664488, … 43.061282]]","{true,true,true,4,5.7464e-8,200}"
"""ARISE-Initiative/robosuite""",188,"[[0.0, 0.002004, … 1.0], [0.785683, 0.784389, … 0.999999], … [30.886659, 30.622994, … 149.880263]]","{true,true,true,4,0.000013,200}"
…,…,…,…
"""zigpy/zha-device-handlers""",1628,"[[0.0, 0.002004, … 1.0], [0.481678, 0.450182, … 0.955854], … [-31.664907, -29.314689, … 51.591479]]","{true,true,true,6,0.000001,200}"
"""ynput/ayon-unreal""",119,"[[0.0, 0.002004, … 1.0], [0.999999, 0.999999, … 0.999999], … [4876.845187, 4876.845187, … 4876.845187]]","{true,true,true,1,1.8029e-7,200}"
"""vyperlang/vyper""",2321,"[[0.0, 0.002004, … 1.0], [0.4216, 0.422553, … 0.999999], … [29.025285, 27.947033, … 360.312345]]","{true,true,true,13,0.00008,200}"


In [7]:
fitted = reviews.filter(pl.col("profile").is_not_null())
probability = np.stack([np.asarray(p[1], float) for p in fitted["profile"].to_list()])
metrics = Stationarity().transform(probability)

discipline = pl.DataFrame(
    {
        "name": fitted["name"].to_list(),
        "n": fitted["n"].to_list(),
        "onset": metrics["onset"],
        "kind": metrics["kind"],
    }
)
discipline

name,n,onset,kind
str,i64,f64,str
"""AMReX-Codes/pyamrex""",373,0.983968,"""transient"""
"""AllenNeuralDynamics/aind-data-…",126,0.977956,"""transient"""
"""ARM-DOE/ACT""",685,0.995992,"""transient"""
"""AMWA-TV/nmos-testing""",462,0.97996,"""transient"""
"""ARISE-Initiative/robosuite""",188,0.490982,"""transient"""
…,…,…,…
"""zigpy/zha-device-handlers""",1628,0.963928,"""transient"""
"""ynput/ayon-unreal""",119,0.0,"""stationary"""
"""vyperlang/vyper""",2321,0.408818,"""transient"""


In [20]:
discipline.filter(pl.col("name").str.starts_with("OpenJobDescription"))

name,n,onset,kind
str,i64,f64,str
"""OpenJobDescription/openjd-adap…",122,0.0,"""stationary"""
"""OpenJobDescription/openjd-mode…",129,0.0,"""stationary"""
"""OpenJobDescription/openjd-sess…",163,0.0,"""stationary"""


In [8]:
def profile_traces(pool: pl.DataFrame, picked: list[str]):
    subset = (
        pool.filter(pl.col("profile").is_not_null())
        .filter(pl.col("name").is_in(picked))
        .select(["name", "n", "profile"])
    )
    colorway = pio.templates["thesis"].layout.colorway
    traces = []
    for idx, r in enumerate(subset.iter_rows(named=True)):
        color = colorway[idx % len(colorway)]
        dom = np.asarray(r["profile"][0], float)
        prob = np.asarray(r["profile"][1], float)
        lower = np.asarray(r["profile"][2], float)
        upper = np.asarray(r["profile"][3], float)
        label = f"{r['name']} (n={r['n']})"
        traces.append(
            go.Scatter(
                x=np.concatenate([dom, dom[::-1]]),
                y=np.concatenate([upper, lower[::-1]]),
                fill="toself",
                fillcolor=color,
                opacity=0.15,
                line=dict(width=0),
                name=label,
                legendgroup=label,
                showlegend=False,
                hoverinfo="skip",
            )
        )
        traces.append(
            go.Scatter(
                x=dom,
                y=prob,
                mode="lines",
                line=dict(color=color),
                name=label,
                legendgroup=label,
                hoverinfo="skip",
            )
        )
    return traces


fitted = reviews.filter(pl.col("profile").is_not_null())
probability = np.stack([np.asarray(p[1], float) for p in fitted["profile"].to_list()])
names_all = np.asarray(fitted["name"].to_list())
n_all = fitted["n"].to_numpy()

metrics = Stationarity().transform(probability)
kinds = metrics["kind"]

print(f"stationary:      {int((kinds == 'stationary').sum())}")
print(f"stationary tail: {int((kinds == 'stationary tail').sum())}")
print(f"transient:       {int((kinds == 'transient').sum())}")

p_start_all = probability[:, 0]
p_end_all = probability[:, -1]

classes = ["stationary", "stationary tail", "transient"]
colorway = pio.templates["thesis"].layout.colorway
color_for = {k: colorway[i] for i, k in enumerate(classes)}

sqs_all = np.sqrt(n_all)
size_all = 4 + 10 * (sqs_all - sqs_all.min()) / (np.ptp(sqs_all) + 1e-12)

selection_state: dict[str, list[str]] = {k: [] for k in classes}


def render():
    picked = [n for k in classes for n in selection_state[k]]
    traces = profile_traces(reviews, picked) if picked else []
    with profile_view.batch_update():
        profile_view.data = []
        for tr in traces:
            profile_view.add_trace(tr)


def make_handler(cls: str, trace_names: np.ndarray):
    def handler(trace, points, selector):
        selection_state[cls] = [trace_names[i] for i in points.point_inds]
        render()

    return handler


traces_data = []
trace_names_per_class = {}
for k in classes:
    mask = kinds == k
    trace_names_per_class[k] = names_all[mask]
    traces_data.append(
        go.Scatter(
            x=p_start_all[mask],
            y=p_end_all[mask],
            mode="markers",
            marker=dict(size=size_all[mask], color=color_for[k], opacity=0.85),
            name=f"{k} (n={int(mask.sum())})",
            hovertext=names_all[mask],
            hovertemplate="%{hovertext}<extra></extra>",
        )
    )

scatter = go.FigureWidget(
    data=traces_data,
    layout=go.Layout(
        dragmode="select",
        height=420,
        xaxis_title="p(0)",
        yaxis_title="p(1)",
        xaxis=dict(range=[0, 1]),
        yaxis=dict(range=[0, 1]),
        showlegend=True,
    ),
)

profile_view = go.FigureWidget(
    layout=go.Layout(
        height=300,
        xaxis_title="Normalized Index",
        yaxis_title="Review Probability",
        yaxis=dict(range=[0, 1], fixedrange=True),
        showlegend=True,
    )
)

for k, trace in zip(classes, scatter.data):
    trace.on_selection(make_handler(k, trace_names_per_class[k]))

VBox([scatter, profile_view])

stationary:      94
stationary tail: 36
transient:       681


    'data': [{'hovertemplate': '%{hovertext}<extra></extra>',
              'hov…

In [18]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

palette = pio.templates["thesis"].layout.colorway

omega = 0.10
epsilon = 0.01

fitted = reviews.filter(pl.col("profile").is_not_null())
probability = np.stack([np.asarray(p[1], float) for p in fitted["profile"].to_list()])
total = probability.shape[0]

ranges = probability.max(axis=1) - probability.min(axis=1)

tolerances = np.linspace(0, 0.1, 200)
tail_share = np.array(
    [
        (
            Stationarity(epsilon=e, omega=omega).transform(probability)["kind"]
            != "transient"
        ).sum()
        / total
        for e in tolerances
    ]
)

onsets = Stationarity(epsilon=epsilon, omega=1.0).transform(probability)["onset"]
onset_grid = np.linspace(0, 0.5, 200)
onset_share = np.array(
    [np.mean(np.isfinite(onsets) & (onsets <= o)) for o in onset_grid]
)

histogram = go.Histogram(
    x=ranges,
    xbins=dict(start=0, end=1, size=0.025),
    marker=dict(color=palette[3]),
    showlegend=False,
)

tolerance_curve = go.Scatter(
    x=tolerances,
    y=tail_share,
    mode="lines",
    line=dict(color=palette[3], width=1),
    showlegend=False,
)

onset_curve = go.Scatter(
    x=onset_grid,
    y=onset_share,
    mode="lines",
    line=dict(color=palette[3], width=1),
    showlegend=False,
)

(
    make_subplots(rows=1, cols=3, horizontal_spacing=0.1)
    .add_trace(histogram, row=1, col=1)
    .add_trace(tolerance_curve, row=1, col=2)
    .add_trace(onset_curve, row=1, col=3)
    .update_xaxes(title_text="Range", range=[0, 1], row=1, col=1)
    .update_yaxes(title_text="Repositories", row=1, col=1)
    .update_xaxes(title_text="Tolerance Threshold", range=[0, 0.1], row=1, col=2)
    .update_yaxes(title_text="Share", range=[0, 0.5], row=1, col=2)
    .update_xaxes(title_text="Onset Threshold", range=[0, 0.5], row=1, col=3)
    .update_yaxes(title_text="Tail Share", range=[0, 0.5], row=1, col=3)
    .update_layout(width=900, height=300)
    .show(config={"toImageButtonOptions": {"format": "svg"}})
)